# DataGuide 재무데이터 분석 (korea_fs_data_from_DG)

`dataguide_fs_analyzer_v1.py` 사용 예시. 모듈은 `Korea_Market/analysis/` 등 노트북과 같은 폴더 또는 sys.path 상에 두면 됩니다.

표시 단위: 억원 (`unit=1e5`, 천원→억원). 분기 인덱스는 `Period('Q')` 로 정규화 (회계분기말 영업일 문제 해소).

In [1]:
# ==========================================================
# PART 1: 환경 + Control Panel + DB
# ==========================================================
import sys
from pathlib import Path
import pandas as pd

def add_repo_path():
    for parent in [Path.cwd()] + list(Path.cwd().parents):
        if (parent / 'DATA').exists():
            if str(parent) not in sys.path:
                sys.path.insert(0, str(parent))
            return parent
    raise FileNotFoundError('DATA 폴더를 찾을 수 없습니다')

PROJECT_ROOT = add_repo_path()
sys.path.insert(0, str(Path.cwd()))

from DATA import config
import dataguide_fs_analyzer_v1 as A

engine = config.get_engine(config.get_db_info())

# ---- Control Panel ----
ASOF    = None      # 기준 분기. 예: '2026Q2'. None → 최근 완전 적재 분기 자동 선택
MARKET  = None      # 'KS' / 'KQ' / None(전체)
TOP_N   = 30        # 스크리너 상위 N개
UNIT    = 1e5       # 천원 → 억원

pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

## 1. 수집된 재무항목 목록 (키워드 검색)

In [2]:
A.search_items(engine)                 # 전체
# A.search_items(engine, '이익')       # 키워드 부분일치
# A.search_items(engine, '차입')

,alias,item_code,indicator,sj_div,n_ticker,min_date,max_date
0,매입채무,M000902006,매입채무(천원),BS,1521,2009-12-30,2026-06-30
1,단기금융상품,M001113350,단기금융상품(금융기관예치금)(천원),BS,1560,2009-12-30,2026-06-30
2,,M001113970,선급비용(천원),BS,1579,2009-12-30,2026-06-30
3,단기차입금,M001121700,단기차입금(*)(천원),BS,1522,2009-12-30,2026-06-30
4,리스부채,M001122020,(금융)리스부채(천원),BS,1508,2009-12-30,2026-06-30
5,,M001122140,미지급금(*)(천원),BS,1582,2009-12-30,2026-06-30
6,,M001122290,미지급비용(천원),BS,1579,2009-12-30,2026-06-30
7,,M001122561,계약부채(천원),BS,722,2017-12-28,2026-06-30
8,,M001122580,선수금(*)(천원),BS,1553,2009-12-30,2026-06-30
9,비지배지분,M001130640,비지배주주지분(천원),BS,1171,2009-12-30,2026-06-30


## 2. 종목별 시계열 (행=분기, 열=항목)
항목은 별칭(`A.ITEM_ALIAS` 참조) / item_code / 항목명 부분일치 모두 가능.

In [3]:
ts = A.get_ts(engine, 'A278470', ['매출액', '매출총이익', '영업이익', '당기순이익', '자본', '영업현금흐름'], start='2023-01-01', unit=UNIT)
print(ts.attrs)
ts

{'ticker': 'A278470', 'company_name': '에이피알'}


,매출액,매출총이익,영업이익,당기순이익,자본,영업현금흐름
q,,,,,,
2023Q1,"1,221.79",906.74,231.89,202.96,"1,291.03",340.18
2023Q2,"1,276.72",976.40,247.85,187.66,"1,486.15",266.15
2023Q3,"1,219.39",905.08,218.60,183.78,"1,686.75",2.04
2023Q4,"1,520.19","1,166.26",343.60,241.05,"1,969.49",470.04
2024Q1,"1,489.28","1,153.34",277.65,240.93,"2,974.76",194.63
2024Q2,"1,554.94","1,188.00",280.11,240.99,"3,087.26",82.80
2024Q3,"1,741.17","1,307.67",272.43,160.07,"2,815.27",200.06
2024Q4,"2,442.15","1,786.99",396.86,433.91,"3,235.24",313.74
2025Q1,"2,660.33","2,008.65",545.68,499.41,"3,455.23",535.10


In [5]:
# 시계열 적재 상태 확인: 결측 분기 체크
ts.isna().sum()

매출액       0
매출총이익     0
영업이익      0
당기순이익     0
자본        0
영업현금흐름    0
dtype: int64

## 3. YoY 성장률 상위 N개

In [6]:
A.yoy_screen(engine, '매출액', n=TOP_N, asof=ASOF, market=MARKET, unit=UNIT)

,ticker,company_name,base_q,t_q,매출액(t-4),매출액(t),growth_%
0,A255440,야스,2025Q2,2026Q2,61.48,872.20,"1,318.79"
1,A402340,SK스퀘어,2025Q2,2026Q2,"18,883.33","196,097.74",938.47
2,A000040,KR모터스,2025Q2,2026Q2,45.36,263.48,480.81
3,A080220,제주반도체,2025Q2,2026Q2,510.67,"2,899.38",467.76
4,A039200,오스코텍,2025Q2,2026Q2,100.17,526.49,425.61
5,A025560,미래산업,2025Q2,2026Q2,68.06,307.98,352.49
6,A054540,삼영엠텍,2025Q2,2026Q2,296.32,"1,313.93",343.42
7,A049950,미래컴퍼니,2025Q2,2026Q2,52.71,233.40,342.77
8,A000680,LS네트웍스,2025Q2,2026Q2,"5,491.57","23,790.93",333.23
9,A092870,엑시콘,2025Q2,2026Q2,75.67,314.69,315.84


In [7]:
A.yoy_screen(engine, '영업이익', n=TOP_N, asof=ASOF, market=MARKET, unit=UNIT, min_base=5e5)   # 기준 영업이익 ≥ 5억원


,ticker,company_name,base_q,t_q,영업이익(t-4),영업이익(t),growth_%
0,A016450,한세예스24홀딩스,2025Q2,2026Q2,5.40,403.93,"7,376.37"
1,A353200,대덕전자,2025Q2,2026Q2,18.66,702.76,"3,665.96"
2,A080220,제주반도체,2025Q2,2026Q2,43.38,"1,218.71","2,709.48"
3,A005950,이수화학,2025Q2,2026Q2,39.09,965.59,"2,370.21"
4,A011070,LG이노텍,2025Q2,2026Q2,113.92,"2,457.54","2,057.24"
5,A003670,포스코퓨처엠,2025Q2,2026Q2,7.73,165.92,"2,045.58"
6,A005930,삼성전자,2025Q2,2026Q2,"46,760.57","894,924.12","1,813.84"
7,A000020,동화약품,2025Q2,2026Q2,6.14,102.90,"1,575.54"
8,A402340,SK스퀘어,2025Q2,2026Q2,"14,011.04","192,354.28","1,272.88"
9,A192440,슈피겐코리아,2025Q2,2026Q2,9.58,125.75,"1,212.82"


## 4. QoQ 성장률 상위 N개

In [ ]:
A.qoq_screen(engine, '매출액', n=TOP_N, asof=ASOF, market=MARKET, unit=UNIT)

In [ ]:
A.qoq_screen(engine, '영업이익', n=TOP_N, asof=ASOF, market=MARKET, unit=UNIT)

## 5. 영업이익 흑자전환

In [ ]:
A.turnaround_screen(engine, basis='yoy', asof=ASOF, market=MARKET, unit=UNIT)   # t-4 적자 → t 흑자

In [ ]:
A.turnaround_screen(engine, basis='qoq', asof=ASOF, market=MARKET, unit=UNIT)   # t-1 적자 → t 흑자

## 6. 재무비율 스크리너
지원: `OPM, NPM, GPM, ROE, ROA, ROIC, 부채비율, 순차입금비율, OCF_margin, FCF_margin`

- 손익 항목은 TTM 합 (`ttm=True`), BS 항목은 기초/기말 평균
- ROIC = 영업이익×(1−유효세율) / (자본 + 차입금 − 현금성자산)  평균
- 자본 ≤ 0, 매출 ≤ 0 기업은 해당 비율 NaN 처리

`compute_ratios()` 를 한 번 호출해 두고 `ratios_df=` 로 넘기면 비율마다 DB 재조회 없이 정렬만 수행.

In [ ]:
ratios = A.compute_ratios(engine, asof=ASOF, market=MARKET, ttm=True, unit=UNIT)
ratios.shape

In [ ]:
A.ratio_screen(engine, 'ROE',  n=TOP_N, ratios_df=ratios)

In [ ]:
A.ratio_screen(engine, 'ROIC', n=TOP_N, ratios_df=ratios)

In [ ]:
A.ratio_screen(engine, 'OPM',  n=TOP_N, ratios_df=ratios)

In [ ]:
A.ratio_screen(engine, '부채비율', n=TOP_N, ratios_df=ratios, ascending=True)   # 낮은 순

In [ ]:
# 특정 종목 비율만 보기
ratios[ratios['ticker'].isin(['A278470', 'A000660', 'A004000'])].T